In [79]:
# ============================================================
# AWS GLUE E-COMMERCE ETL
# Historical CSV + Ordering API JSON
# S3 Curated + Amazon Redshift
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from functools import reduce

In [80]:
# ============================================================
# 1. SPARK SESSION
# ============================================================

spark = SparkSession.builder \
    .appName("Ecommerce-Historical-API-ETL") \
    .getOrCreate()

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

BUCKET = "aws-ecommerce-data-wael-2026"

# Historical data
RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"

# API incremental data
API_RAW_PATH = f"s3://{BUCKET}/api_raw/orders"

# Curated data
PROCESSED_PATH = f"s3://{BUCKET}/curated/ecommerce"

In [1]:
# # ============================================================
# # REDSHIFT CONFIGURATION
# # ============================================================

# REDSHIFT_CONNECTION = "ecommerce-redshift-connection"

# REDSHIFT_DATABASE = "dev"

# REDSHIFT_SCHEMA = "public"

# # S3 temporary directory used by Spark/Glue
# REDSHIFT_TEMP_DIR = (
#     f"s3://{BUCKET}/redshift-temp/"
# )

In [82]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def clean_column_names(df):

    for column in df.columns:

        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        df = df.withColumnRenamed(
            column,
            new_name
        )

    return df


def read_csv(filename):

    path = f"{RAW_PATH}/{filename}"

    print("Reading:", path)

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .csv(path)
    )


def write_parquet(df, table_name):

    output_path = (
        f"{PROCESSED_PATH}/{table_name}"
    )

    print(
        f"Writing {table_name} -> {output_path}"
    )

    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .save(output_path)
    )

In [83]:
# ============================================================
# 4. READ HISTORICAL DATA
# ============================================================

print("============================================")
print("READING HISTORICAL DATA")
print("============================================")


customers = read_csv("customers.csv")

categories = read_csv("categories.csv")

products = read_csv("products.csv")

departments = read_csv("departments.csv")

employees = read_csv("employees.csv")

suppliers = read_csv("suppliers.csv")

orders = read_csv("orders.csv")

order_details = read_csv("order_details.csv")

payments = read_csv("payments.csv")

product_suppliers = read_csv(
    "product_suppliers.csv"
)

shippers = read_csv("shippers.csv")

shipments = read_csv("shipments.csv")

READING HISTORICAL DATA
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/customers.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/categories.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/products.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/departments.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/employees.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/suppliers.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/orders.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/order_details.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/payments.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/product_suppliers.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/shippers.csv
Reading: s3://aws-ecommerce-data-omar-2026/raw/ecommerce/shipments.csv


In [84]:
customers.printSchema()                  
categories.printSchema()
products.printSchema()
departments.printSchema()
employees.printSchema()
suppliers.printSchema()
orders.printSchema()
order_details.printSchema()
payments.printSchema()
product_suppliers.printSchema()
shippers.printSchema()
shipments.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

root
 |-- CategoryID: integer (nullable = true)
 |-- CategoryName: string (nullable = true)

root
 |-- ProductID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Stock: integer (nullable = true)

root
 |-- DepartmentID: integer (nullable = true)
 |-- DepartmentName: string (nullable = true)

root
 |-- EmployeeID: integer (nullable = true)
 |-- ManagerID: integer (nullable = true)
 |-- DepartmentID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Salary: double

In [6]:
# ============================================================
# 5. CLEAN COLUMN NAMES
# ============================================================

print("============================================")
print("CLEANING COLUMN NAMES")
print("============================================")


customers = clean_column_names(customers)

categories = clean_column_names(categories)

products = clean_column_names(products)

departments = clean_column_names(departments)

employees = clean_column_names(employees)

suppliers = clean_column_names(suppliers)

orders = clean_column_names(orders)

order_details = clean_column_names(order_details)

payments = clean_column_names(payments)

product_suppliers = clean_column_names(
    product_suppliers
)

shippers = clean_column_names(shippers)

shipments = clean_column_names(shipments)

CLEANING COLUMN NAMES


In [7]:
# ============================================================
# 6. READ API JSON
# ============================================================

print("============================================")
print("READING API JSON")
print("============================================")

api_json = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API JSON schema:")

api_json.printSchema()

Py4JJavaError: An error occurred while calling o229.json.
: org.apache.hadoop.mapreduce.lib.input.InvalidInputException: Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000001.json
Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000002.json
Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000003.json
Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000004.json
Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000005.json
Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000006.json
Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000007.json
Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000008.json
Input path does not exist: s3://aws-ecommerce-data-omar-2026/api_raw/orders/order_2000009.json
Input path does not exis

In [ ]:
import boto3

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket="aws-ecommerce-data-wael-2026",
    Prefix="api_raw/orders/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

api_raw/orders/
api_raw/orders/order_2000001.json
api_raw/orders/order_2000002.json
api_raw/orders/order_2000003.json
api_raw/orders/order_2000004.json
api_raw/orders/order_2000005.json
api_raw/orders/order_2000006.json
api_raw/orders/order_2000007.json
api_raw/orders/order_2000008.json
api_raw/orders/order_2000009.json
api_raw/orders/order_2000010.json
api_raw/orders/order_2000011.json
api_raw/orders/order_2000012.json
api_raw/orders/order_2000013.json
api_raw/orders/order_2000014.json
api_raw/orders/order_2000015.json
api_raw/orders/order_2000016.json
api_raw/orders/order_2000017.json
api_raw/orders/order_2000018.json
api_raw/orders/order_2000019.json
api_raw/orders/order_2000020.json
api_raw/orders/order_2000021.json
api_raw/orders/order_2000022.json
api_raw/orders/order_2000023.json
api_raw/orders/order_2000024.json
api_raw/orders/order_2000025.json
api_raw/orders/order_2000026.json
api_raw/orders/order_2000027.json
api_raw/orders/order_2000028.json
api_raw/orders/order_2000029.jso

In [15]:
# ============================================================
# 6. READ API JSON
# ============================================================

print("============================================")
print("READING API JSON")
print("============================================")

api_json = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API JSON schema:")

api_json.printSchema()

READING API JSON
API JSON schema:
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |

In [ ]:
from pyspark.sql import functions as F

BUCKET = "aws-ecommerce-data-wael-2026"

API_RAW_PATH = f"s3://{BUCKET}/api_raw/orders/"

api_orders = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API records:", api_orders.count())

api_orders.printSchema()

API records: 110
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- PaymentDate: s

In [19]:
api_orders_df = api_orders.select(
    
    F.col("order.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("order.CustomerID")
        .cast("long")
        .alias("customerid"),

    F.to_date(
        F.col("order.OrderDate")
    ).alias("orderdate"),

    F.upper(
        F.trim(
            F.col("order.Status")
        )
    ).alias("status"),

    # API metadata
    F.col("metadata.event_id")
        .alias("event_id"),

    F.to_timestamp(
        F.col("metadata.event_timestamp")
    ).alias("event_timestamp"),

    F.col("metadata.event_type")
        .alias("event_type"),

    F.col("metadata.source")
        .alias("source")
)

api_orders_df.show(10, truncate=False)

+-------+----------+----------+---------+------------------------------------+--------------------------+-------------+------------+
|orderid|customerid|orderdate |status   |event_id                            |event_timestamp           |event_type   |source      |
+-------+----------+----------+---------+------------------------------------+--------------------------+-------------+------------+
|2000066|454494    |2026-09-19|CONFIRMED|8ebd3017-4eec-4dec-88d3-2d8770701cde|2026-09-19 20:52:27.607969|ORDER_CREATED|ordering_api|
|2000070|423391    |2026-09-19|CONFIRMED|b76452f0-c298-45f6-9694-f3dab82b458b|2026-09-19 20:52:27.610965|ORDER_CREATED|ordering_api|
|2000064|45340     |2026-09-19|CONFIRMED|9e11b34a-008b-443f-b832-8fa2c4f7c3f4|2026-09-19 20:52:27.606965|ORDER_CREATED|ordering_api|
|2000067|19051     |2026-09-19|CONFIRMED|b63a3648-09d1-479a-8343-bbbf6772c316|2026-09-19 20:52:27.608965|ORDER_CREATED|ordering_api|
|2000063|434201    |2026-09-19|CONFIRMED|5ece1678-2ec9-49a1-9c01-9d82

In [77]:
api_order_details_df = (
    api_orders
    .select(
        
        F.col("order.OrderID")
            .cast("long")
            .alias("orderid"),

        F.explode(
            F.col("order_details")
        ).alias("detail")
    )
)
api_order_details_df.show()

+-------+--------------------+
|orderid|              detail|
+-------+--------------------+
|2000066|{0.0, 5000161, 20...|
|2000066|{15.0, 5000162, 2...|
|2000066|{10.0, 5000163, 2...|
|2000066|{10.0, 5000164, 2...|
|2000070|{10.0, 5000176, 2...|
|2000070|{0.0, 5000177, 20...|
|2000070|{0.0, 5000178, 20...|
|2000070|{15.0, 5000179, 2...|
|2000064|{5.0, 5000153, 20...|
|2000064|{5.0, 5000154, 20...|
|2000064|{15.0, 5000155, 2...|
|2000064|{15.0, 5000156, 2...|
|2000067|{10.0, 5000165, 2...|
|2000067|{0.0, 5000166, 20...|
|2000067|{0.0, 5000167, 20...|
|2000067|{10.0, 5000168, 2...|
|2000063|{10.0, 5000149, 2...|
|2000063|{5.0, 5000150, 20...|
|2000063|{5.0, 5000151, 20...|
|2000063|{0.0, 5000152, 20...|
+-------+--------------------+
only showing top 20 rows


In [78]:
api_order_details_df = api_order_details_df.select(

    "orderid",

    F.col("detail.OrderDetailID")
        .cast("long")
        .alias("orderdetailid"),

    F.col("detail.ProductID")
        .cast("long")
        .alias("productid"),

    F.col("detail.Quantity")
        .cast("long")
        .alias("quantity"),

    F.col("detail.UnitPrice")
        .cast("decimal(12,2)")
        .alias("unitprice"),

    F.col("detail.Discount")
        .cast("decimal(12,2)")
        .alias("discount")
)
api_order_details_df.show()

+-------+-------------+---------+--------+---------+--------+
|orderid|orderdetailid|productid|quantity|unitprice|discount|
+-------+-------------+---------+--------+---------+--------+
|2000066|      5000161|     1575|       3|   925.38|    0.00|
|2000066|      5000162|     2228|       2|    14.98|   15.00|
|2000066|      5000163|     1472|       2|   963.14|   10.00|
|2000066|      5000164|     2453|       5|   766.60|   10.00|
|2000070|      5000176|     4462|       3|   347.25|   10.00|
|2000070|      5000177|     4634|       1|   243.95|    0.00|
|2000070|      5000178|     9225|       1|   244.43|    0.00|
|2000070|      5000179|     7473|       4|   974.45|   15.00|
|2000064|      5000153|     4386|       3|   184.32|    5.00|
|2000064|      5000154|     1180|       4|   228.83|    5.00|
|2000064|      5000155|     1270|       1|   640.38|   15.00|
|2000064|      5000156|     9278|       1|   529.57|   15.00|
|2000067|      5000165|     3533|       4|   154.68|   10.00|
|2000067

In [24]:
api_payments_df = api_orders.select(

    F.col("payment.PaymentID")
        .cast("long")
        .alias("paymentid"),

    F.col("payment.OrderID")
        .cast("long")
        .alias("orderid"),

    F.trim(
        F.col("payment.PaymentMethod")
    ).alias("paymentmethod"),

    F.to_date(
        F.col("payment.PaymentDate")
    ).alias("paymentdate"),

    F.col("payment.Amount")
        .cast("decimal(12,2)")
        .alias("amount"),

    F.upper(
        F.trim(
            F.col("payment.PaymentStatus")
        )
    ).alias("paymentstatus")
)

api_payments_df.show(10)

+---------+-------+----------------+-----------+--------+-------------+
|paymentid|orderid|   paymentmethod|paymentdate|  amount|paymentstatus|
+---------+-------+----------------+-----------+--------+-------------+
|  2000066|2000066|Cash on Delivery| 2026-09-19| 9152.85|    COMPLETED|
|  2000070|2000070|Cash on Delivery| 2026-09-19| 5452.55|    COMPLETED|
|  2000064|2000064|Cash on Delivery| 2026-09-19| 2773.81|    COMPLETED|
|  2000067|2000067|Cash on Delivery| 2026-09-19| 2717.54|    COMPLETED|
|  2000063|2000063|   Bank Transfer| 2026-09-19|11161.52|    COMPLETED|
|  2000079|2000079|Cash on Delivery| 2026-09-19| 3734.62|    COMPLETED|
|  2000100|2000100|Cash on Delivery| 2026-09-19| 3939.16|    COMPLETED|
|  2000109|2000109|Cash on Delivery| 2026-09-19| 6176.38|    COMPLETED|
|  2000065|2000065|   Bank Transfer| 2026-09-19| 4114.77|    COMPLETED|
|  2000093|2000093|Cash on Delivery| 2026-09-19| 1316.71|    COMPLETED|
+---------+-------+----------------+-----------+--------+-------

In [25]:
api_shipments_df = api_orders.select(

    F.col("shipment.ShipmentID")
        .cast("long")
        .alias("shipmentid"),

    F.col("shipment.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("shipment.ShipperID")
        .cast("long")
        .alias("shipperid"),

    F.to_date(
        F.col("shipment.ShipDate")
    ).alias("shipdate"),

    F.to_date(
        F.col("shipment.DeliveryDate")
    ).alias("deliverydate")
)

api_shipments_df.show(10)

+----------+-------+---------+----------+------------+
|shipmentid|orderid|shipperid|  shipdate|deliverydate|
+----------+-------+---------+----------+------------+
|    500064|2000066|      416|2026-09-20|  2026-09-25|
|    500068|2000070|      558|2026-09-20|  2026-09-24|
|    500062|2000064|      606|2026-09-20|  2026-09-23|
|    500065|2000067|      472|2026-09-20|  2026-09-22|
|    500061|2000063|      378|2026-09-20|  2026-09-22|
|    500077|2000079|      118|2026-09-20|  2026-09-23|
|    500098|2000100|      730|2026-09-20|  2026-09-24|
|    500106|2000109|      407|2026-09-20|  2026-09-23|
|    500063|2000065|      638|2026-09-20|  2026-09-22|
|    500091|2000093|      687|2026-09-20|  2026-09-22|
+----------+-------+---------+----------+------------+
only showing top 10 rows


In [28]:
api_order_summary_df = api_orders.select(

    F.col("order.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("summary.subtotal")
        .cast("decimal(12,2)")
        .alias("subtotal"),

    F.col("summary.discount")
        .cast("decimal(12,2)")
        .alias("discount"),

    F.col("summary.tax_rate")
        .cast("decimal(5,2)")
        .alias("tax_rate"),

    F.col("summary.tax_amount")
        .cast("decimal(12,2)")
        .alias("tax_amount"),

    F.col("summary.shipping_cost")
        .cast("decimal(12,2)")
        .alias("shipping_cost"),

    F.col("summary.total_amount")
        .cast("decimal(12,2)")
        .alias("total_amount")
)
api_order_summary_df.show(10)

+-------+--------+--------+--------+----------+-------------+------------+
|orderid|subtotal|discount|tax_rate|tax_amount|shipping_cost|total_amount|
+-------+--------+--------+--------+----------+-------------+------------+
|2000066| 7984.96|  580.42|    0.14|   1117.89|        50.00|     9152.85|
|2000070| 4739.08|  688.85|    0.14|    663.47|        50.00|     5452.55|
|2000064| 2389.31|  248.92|    0.14|    334.50|        50.00|     2773.81|
|2000067| 2339.95|  156.53|    0.14|    327.59|        50.00|     2717.54|
|2000063| 9746.95|  400.70|    0.14|   1364.57|        50.00|    11161.52|
|2000079| 3232.12|  118.36|    0.14|    452.50|        50.00|     3734.62|
|2000100| 3411.54|  306.54|    0.14|    477.62|        50.00|     3939.16|
|2000109| 5374.02|  297.77|    0.14|    752.36|        50.00|     6176.38|
|2000065| 3565.59|  263.60|    0.14|    499.18|        50.00|     4114.77|
|2000093| 1111.15|   65.12|    0.14|    155.56|        50.00|     1316.71|
+-------+--------+-------

In [29]:
historical_orders = orders.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
)

In [30]:
new_orders = api_orders_df.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
)

In [32]:
all_orders = (
    historical_orders
    .unionByName(new_orders)
    .dropDuplicates(["orderid"])
)
all_orders.show(10)

+-------+----------+-------------------+----------+
|orderid|customerid|          orderdate|    status|
+-------+----------+-------------------+----------+
|     28|     71742|2026-08-01 00:00:00| Cancelled|
|     29|    272105|2024-03-28 00:00:00|Processing|
|     30|    352778|2024-12-28 00:00:00| Cancelled|
|     33|     67214|2026-03-31 00:00:00|   Pending|
|     93|     55389|2025-12-16 00:00:00|Processing|
|    151|    351912|2025-06-19 00:00:00| Delivered|
|    171|    108537|2024-04-26 00:00:00|   Pending|
|    200|    463755|2024-08-25 00:00:00|Processing|
|    208|    293998|2026-08-16 00:00:00|Processing|
|    221|    419094|2026-02-03 00:00:00| Cancelled|
+-------+----------+-------------------+----------+
only showing top 10 rows


In [33]:
print("Historical orders:", historical_orders.count())
print("API orders:", new_orders.count())
print("Combined orders:", all_orders.count())

Historical orders: 2000000
API orders: 110
Combined orders: 2000110


In [86]:
historical_orders_details = order_details.select(
    "OrderDetailID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "Discount"
)

In [87]:
new_orders_details = api_order_details_df.select(
    "OrderDetailID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "Discount"
)

In [88]:
all_orders_details = (
    historical_orders_details
    .unionByName(new_orders_details)
    .dropDuplicates(["OrderDetailID"])
)
all_orders_details.show(10)

+-------------+-------+---------+--------+---------+--------+
|OrderDetailID|OrderID|ProductID|Quantity|UnitPrice|Discount|
+-------------+-------+---------+--------+---------+--------+
|           28|1135417|     7156|       3|  2423.09|    5.00|
|           29|1941313|     8432|       8|  1981.19|    0.00|
|           30|1711404|     2125|      10|   421.11|    0.00|
|           33| 454099|     2748|       2|  1000.65|   10.00|
|           93|  67034|     8522|       1|  1666.62|    5.00|
|          151| 373422|     3814|       9|    270.0|   10.00|
|          171|1538435|     7832|       9|   281.78|    0.00|
|          200|1212882|     8626|       1|  1908.14|    0.00|
|          208|  32331|      502|       3|  1424.23|    5.00|
|          221| 399276|     5755|       1|  2134.98|    0.00|
+-------------+-------+---------+--------+---------+--------+
only showing top 10 rows


In [89]:
print("Historical orders_details:", historical_orders_details.count())
print("API orders_details:", new_orders_details.count())
print("Combined orders_details:", all_orders_details.count())

Historical orders: 5000000
API orders: 274
Combined orders: 5000274


In [143]:
historical_payments = payments.select(
    "paymentid",
    "orderid",
    "paymentmethod",
    "paymentdate",
    "amount"
)

In [144]:
new_orders_payments = api_payments_df.select(
    "paymentid",
    "orderid",
    "paymentmethod",
    "paymentdate",
    "amount"
)

In [147]:
all_payments = (
    historical_payments
    .unionByName(new_orders_payments)
    .dropDuplicates(["paymentid"])
)
all_payments.show(10)

+---------+-------+-------------+-------------------+-------+
|paymentid|orderid|paymentmethod|        paymentdate| amount|
+---------+-------+-------------+-------------------+-------+
|       28|1608346|       PayPal|2024-03-16 00:00:00| 129.14|
|       29| 234002|         Cash|2024-01-06 00:00:00|  71.76|
|       30| 875670|   Debit Card|2024-10-26 00:00:00|2674.33|
|       33| 485893|         Cash|2025-01-23 00:00:00|3987.04|
|       93|1407880|         Cash|2025-03-09 00:00:00|2618.59|
|      151|1372269|         Cash|2026-06-28 00:00:00|2209.35|
|      171| 511885|Bank Transfer|2025-06-22 00:00:00|3296.48|
|      200|1114293|   Debit Card|2025-03-18 00:00:00|  454.0|
|      208|1866528|  Credit Card|2024-07-06 00:00:00|3813.99|
|      221| 898861|   Debit Card|2025-01-03 00:00:00|1964.67|
+---------+-------+-------------+-------------------+-------+
only showing top 10 rows


In [148]:
print("Historical payments:", historical_payments.count())
print("API payments:", new_orders_payments.count())
print("Combined payments:", all_payments.count())

Historical payments: 2000000
API payments: 110
Combined payments: 2000110


In [98]:
historical_shipments = shipments.select(
   "ShipmentID",
    "OrderID",
    "ShipperID",
    "ShipDate",
    "DeliveryDate"
)

In [119]:
new_orders_shipments = api_shipments_df.select(
   "ShipmentID",
    "OrderID",
    "ShipperID",
    "ShipDate",
    "DeliveryDate"
)

In [120]:
all_shipments = (
    historical_shipments
    .unionByName(new_orders_shipments)
    .dropDuplicates(["ShipmentID"])
)
all_shipments.show(10)

+----------+-------+---------+-------------------+-------------------+
|ShipmentID|OrderID|ShipperID|           ShipDate|       DeliveryDate|
+----------+-------+---------+-------------------+-------------------+
|        28| 927753|      526|2025-10-23 00:00:00|2025-10-25 00:00:00|
|        29|1023284|      121|2026-04-23 00:00:00|2026-05-03 00:00:00|
|        30| 430775|      650|2026-01-14 00:00:00|2026-01-22 00:00:00|
|        33|  16018|      529|2025-10-24 00:00:00|2025-10-25 00:00:00|
|        93|1585580|      795|2025-02-06 00:00:00|2025-02-08 00:00:00|
|       151| 925600|      493|2025-02-10 00:00:00|2025-02-20 00:00:00|
|       171|1133526|      442|2024-02-12 00:00:00|2024-02-13 00:00:00|
|       200|1618444|       46|2025-09-27 00:00:00|2025-09-28 00:00:00|
|       208|1989748|      781|2025-04-02 00:00:00|2025-04-11 00:00:00|
|       221|1721171|      194|2024-05-23 00:00:00|2024-05-30 00:00:00|
+----------+-------+---------+-------------------+-------------------+
only s

In [121]:
print("Historical shipments:", historical_shipments.count())
print("API shipments:", new_orders_shipments.count())
print("Combined shipments:", all_shipments.count())

Historical shipments: 500000
API shipments: 110
Combined shipments: 500108


In [122]:
all_shipments.groupBy("ShipmentID") \
    .count() \
    .filter("count > 1") \
    .show()

+----------+-----+
|ShipmentID|count|
+----------+-----+
+----------+-----+


In [126]:
all_shipments.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in all_shipments.columns
]).show()

+----------+-------+---------+--------+------------+
|ShipmentID|OrderID|ShipperID|ShipDate|DeliveryDate|
+----------+-------+---------+--------+------------+
|         1|      1|        1|       1|           1|
+----------+-------+---------+--------+------------+


In [127]:
all_shipments = all_shipments.dropna(how="all")

In [128]:
print("Historical shipments:", historical_shipments.count())
print("API shipments:", new_orders_shipments.count())
print("Combined shipments:", all_shipments.count())

Historical shipments: 500000
API shipments: 110
Combined shipments: 500107


In [129]:
all_orders_details = (
    all_orders_details

    # Make sure types are consistent
    .withColumn("orderdetailid", F.col("orderdetailid").cast("long"))
    .withColumn("orderid", F.col("orderid").cast("long"))
    .withColumn("productid", F.col("productid").cast("long"))
    .withColumn("quantity", F.col("quantity").cast("long"))
    .withColumn("unitprice", F.col("unitprice").cast("decimal(12,2)"))
    .withColumn("discount", F.col("discount").cast("decimal(12,2)"))

    # Handle negative values
    .withColumn(
        "quantity",
        F.when(F.col("quantity") < 0, 0)
         .otherwise(F.col("quantity"))
    )

    .withColumn(
        "unitprice",
        F.when(F.col("unitprice") < 0, 0)
         .otherwise(F.col("unitprice"))
    )

    .withColumn(
        "discount",
        F.when(F.col("discount") < 0, 0)
         .otherwise(F.col("discount"))
    )

    # Recalculate amounts
    .withColumn(
        "gross_amount",
        F.col("quantity") * F.col("unitprice")
    )

    .withColumn(
        "discount_amount",
        F.col("gross_amount") * F.col("discount") / 100
    )

    .withColumn(
        "total_amount",
        F.col("gross_amount") - F.col("discount_amount")
    )
)

In [130]:
all_orders_details.show(2)

+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
|orderdetailid|orderid|productid|quantity|unitprice|discount|gross_amount|discount_amount|total_amount|
+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
|           28|1135417|     7156|       3|  2423.09|    5.00|     7269.27|     363.463500| 6905.806500|
|           29|1941313|     8432|       8|  1981.19|    0.00|    15849.52|       0.000000|15849.520000|
+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
only showing top 2 rows


In [131]:
all_orders_details.count()

5000274


In [34]:
all_api_orders = (
    api_orders_df
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
)
all_api_orders.show(10)

+-------+----------+----------+---------+--------------------+--------------------+-------------+------------+----+-----+
|orderid|customerid| orderdate|   status|            event_id|     event_timestamp|   event_type|      source|year|month|
+-------+----------+----------+---------+--------------------+--------------------+-------------+------------+----+-----+
|2000066|    454494|2026-09-19|CONFIRMED|8ebd3017-4eec-4de...|2026-09-19 20:52:...|ORDER_CREATED|ordering_api|2026|    9|
|2000070|    423391|2026-09-19|CONFIRMED|b76452f0-c298-45f...|2026-09-19 20:52:...|ORDER_CREATED|ordering_api|2026|    9|
|2000064|     45340|2026-09-19|CONFIRMED|9e11b34a-008b-443...|2026-09-19 20:52:...|ORDER_CREATED|ordering_api|2026|    9|
|2000067|     19051|2026-09-19|CONFIRMED|b63a3648-09d1-479...|2026-09-19 20:52:...|ORDER_CREATED|ordering_api|2026|    9|
|2000063|    434201|2026-09-19|CONFIRMED|5ece1678-2ec9-49a...|2026-09-19 20:52:...|ORDER_CREATED|ordering_api|2026|    9|
|2000079|    317425|2026

In [38]:
API_CURATED_PATH = (
    f"s3://{BUCKET}/curated/ecommerce/api_orders"
)

(
    all_api_orders
    .write
    .mode("append")
    .format("parquet")
    .partitionBy("year", "month")
    .option("compression", "snappy")
    .save(API_CURATED_PATH)
)

In [132]:
dim_customer = (
    customers
    .select(
        F.col("customerid").cast("long").alias("customer_id"),
        F.trim("firstname").alias("first_name"),
        F.trim("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.trim("firstname"),
            F.trim("lastname")
        ).alias("full_name"),
        F.lower(F.trim("email")).alias("email"),
        F.trim("city").alias("city"),
        F.trim("country").alias("country"),
        F.to_date("registrationdate").alias("registration_date")
    )
    .dropDuplicates(["customer_id"])
)

In [133]:
dim_category = (
    categories
    .select(
        F.col("categoryid").cast("long").alias("category_id"),
        F.trim("categoryname").alias("category_name")
    )
    .dropDuplicates(["category_id"])
)

In [134]:
dim_department = (
    departments
    .select(
        F.col("departmentid").cast("long").alias("department_id"),
        F.trim("departmentname").alias("department_name")
    )
    .dropDuplicates(["department_id"])
)

In [135]:
dim_product = (
    products
    .select(
        F.col("productid").cast("long").alias("product_id"),
        F.col("categoryid").cast("long").alias("category_id"),
        F.trim("productname").alias("product_name"),
        F.trim("brand").alias("brand"),
        F.col("price").cast("decimal(12,2)").alias("price"),
        F.col("cost").cast("decimal(12,2)").alias("cost"),
        F.col("stock").cast("int").alias("stock")
    )
    .dropDuplicates(["product_id"])
)

In [136]:
dim_supplier = (
    suppliers
    .select(
        F.col("supplierid").cast("long").alias("supplier_id"),
        F.trim("suppliername").alias("supplier_name"),
        F.trim("country").alias("country")
    )
    .dropDuplicates(["supplier_id"])
)

In [137]:
dim_employee = (
    employees
    .select(
        F.col("employeeid").cast("long").alias("employee_id"),
        F.col("managerid").cast("long").alias("manager_id"),
        F.col("departmentid").cast("long").alias("department_id"),
        F.trim("firstname").alias("first_name"),
        F.trim("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.trim("firstname"),
            F.trim("lastname")
        ).alias("full_name"),
        F.col("salary").cast("decimal(12,2)").alias("salary"),
        F.to_date("hiredate").alias("hire_date")
    )
    .dropDuplicates(["employee_id"])
)

In [138]:
dim_shipper = (
    shippers
    .select(
        F.col("shipperid").cast("long").alias("shipper_id"),
        F.trim("companyname").alias("company_name")
    )
    .dropDuplicates(["shipper_id"])
)

In [140]:
date_range = (
    all_orders
    .select(
        F.to_date("orderdate").alias("date")
    )
    .filter(F.col("date").isNotNull())
    .agg(
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date")
    )
)

In [141]:
dim_date = (
    date_range
    .select(
        F.explode(
            F.sequence(
                F.col("min_date"),
                F.col("max_date"),
                F.expr("interval 1 day")
            )
        ).alias("date")
    )
)

In [142]:
dim_date = (
    dim_date
    .withColumn(
        "date_id",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn(
        "month_name",
        F.date_format("date", "MMMM")
    )
    .withColumn(
        "day",
        F.dayofmonth("date")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("date")
    )
    .withColumn(
        "day_name",
        F.date_format("date", "EEEE")
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("date")
    )
)

In [149]:
payment_methods = (
    all_payments
    .select(
        F.upper(
            F.trim("paymentmethod")
        ).alias("payment_method")
    )
    .filter(
        F.col("payment_method").isNotNull()
    )
    .distinct()
)

In [150]:
window_payment = Window.orderBy("payment_method")

dim_payment_method = (
    payment_methods
    .withColumn(
        "payment_method_id",
        F.row_number().over(window_payment)
    )
    .select(
        "payment_method_id",
        "payment_method"
    )
)

In [151]:
order_statuses = (
    all_orders
    .select(
        F.upper(
            F.trim("status")
        ).alias("order_status")
    )
    .filter(
        F.col("order_status").isNotNull()
    )
    .distinct()
)

In [152]:
window_status = Window.orderBy("order_status")

dim_order_status = (
    order_statuses
    .withColumn(
        "order_status_id",
        F.row_number().over(window_status)
    )
    .select(
        "order_status_id",
        "order_status"
    )
)

In [153]:
fact_order = (
    all_orders
    .select(
        F.col("orderid").cast("long").alias("order_id"),
        F.col("customerid").cast("long").alias("customer_id"),
        F.to_date("orderdate").alias("order_date"),
        F.date_format(
            F.to_date("orderdate"),
            "yyyyMMdd"
        ).cast("int").alias("date_id"),
        F.upper(
            F.trim("status")
        ).alias("order_status")
    )
)

In [154]:
fact_order = (
    fact_order
    .join(
        dim_order_status,
        fact_order.order_status ==
        dim_order_status.order_status,
        "left"
    )
    .select(
        "order_id",
        "customer_id",
        "date_id",
        "order_date",
        "order_status_id"
    )
)

In [155]:
fact_order_detail = (
    all_orders_details
    .select(
        F.col("orderdetailid")
            .cast("long")
            .alias("order_detail_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.col("productid")
            .cast("long")
            .alias("product_id"),

        F.col("quantity")
            .cast("int")
            .alias("quantity"),

        F.col("unitprice")
            .cast("decimal(12,2)")
            .alias("unit_price"),

        F.col("discount")
            .cast("decimal(12,2)")
            .alias("discount"),

        F.col("gross_amount")
            .cast("decimal(14,2)")
            .alias("gross_amount"),

        F.col("discount_amount")
            .cast("decimal(14,2)")
            .alias("discount_amount"),

        F.col("total_amount")
            .cast("decimal(14,2)")
            .alias("total_amount")
    )
    .dropDuplicates(["order_detail_id"])
)

In [156]:
fact_order_detail.show(5)

+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+
|order_detail_id|order_id|product_id|quantity|unit_price|discount|gross_amount|discount_amount|total_amount|
+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+
|             28| 1135417|      7156|       3|   2423.09|    5.00|     7269.27|         363.46|     6905.81|
|             29| 1941313|      8432|       8|   1981.19|    0.00|    15849.52|           0.00|    15849.52|
|             30| 1711404|      2125|      10|    421.11|    0.00|     4211.10|           0.00|     4211.10|
|             33|  454099|      2748|       2|   1000.65|   10.00|     2001.30|         200.13|     1801.17|
|             93|   67034|      8522|       1|   1666.62|    5.00|     1666.62|          83.33|     1583.29|
+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+
only showing top 5 

In [157]:
fact_payment_base = (
    all_payments
    .select(
        F.col("paymentid")
            .cast("long")
            .alias("payment_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.to_date("paymentdate")
            .alias("payment_date"),

        F.date_format(
            F.to_date("paymentdate"),
            "yyyyMMdd"
        )
        .cast("int")
        .alias("date_id"),

        F.upper(
            F.trim("paymentmethod")
        )
        .alias("payment_method"),

        F.col("amount")
            .cast("decimal(14,2)")
            .alias("amount")
    )
)

In [158]:
fact_payment = (
    fact_payment_base
    .join(
        dim_payment_method,
        fact_payment_base.payment_method ==
        dim_payment_method.payment_method,
        "left"
    )
    .select(
        "payment_id",
        "order_id",
        "date_id",
        "payment_date",
        "payment_method_id",
        "amount"
    )
)

In [159]:
fact_shipment = (
    all_shipments
    .select(
        F.col("shipmentid")
            .cast("long")
            .alias("shipment_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.col("shipperid")
            .cast("long")
            .alias("shipper_id"),

        F.to_date("shipdate")
            .alias("ship_date"),

        F.to_date("deliverydate")
            .alias("delivery_date")
    )
    .withColumn(
        "shipping_days",
        F.datediff(
            F.col("delivery_date"),
            F.col("ship_date")
        )
    )
)

In [161]:
fact_shipment.show(2)

+-----------+--------+----------+----------+-------------+-------------+
|shipment_id|order_id|shipper_id| ship_date|delivery_date|shipping_days|
+-----------+--------+----------+----------+-------------+-------------+
|         28|  927753|       526|2025-10-23|   2025-10-25|            2|
|         29| 1023284|       121|2026-04-23|   2026-05-03|           10|
+-----------+--------+----------+----------+-------------+-------------+
only showing top 2 rows


In [ ]:
customer_sales_base = (
    fact_order_detail
    .join(
        fact_order.select(
            "order_id",
            "customer_id",
            "order_date"
        ),
        "order_id",
        "inner"
    )
)

In [162]:
fact_customer_sales = (
    customer_sales_base
    .groupBy("customer_id")
    .agg(
        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("total_quantity"),

        F.sum(
            "gross_amount"
        ).alias("gross_sales"),

        F.sum(
            "discount_amount"
        ).alias("total_discount"),

        F.sum(
            "total_amount"
        ).alias("total_sales"),

        F.avg(
            "total_amount"
        ).alias("average_order_line"),

        F.min(
            "order_date"
        ).alias("first_order_date"),

        F.max(
            "order_date"
        ).alias("last_order_date")
    )
)

In [163]:
fact_customer_sales = (
    fact_customer_sales
    .withColumn(
        "customer_segment",
        F.when(
            F.col("total_sales") >= 10000,
            "VIP"
        )
        .when(
            F.col("total_sales") >= 5000,
            "PREMIUM"
        )
        .when(
            F.col("total_sales") >= 1000,
            "REGULAR"
        )
        .otherwise(
            "LOW_VALUE"
        )
    )
)

In [166]:
fact_customer_sales.show(1)

+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
|customer_id|total_orders|total_quantity|gross_sales|total_discount|total_sales|average_order_line|first_order_date|last_order_date|customer_segment|
+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
|      71742|           8|           107|  134669.14|       8971.96|  125697.18|       6615.641053|      2024-07-28|     2026-08-01|             VIP|
+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
only showing top 1 row


In [164]:
fact_product_sales = (
    fact_order_detail
    .groupBy("product_id")
    .agg(
        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("units_sold"),

        F.sum(
            "gross_amount"
        ).alias("gross_sales"),

        F.sum(
            "discount_amount"
        ).alias("total_discount"),

        F.sum(
            "total_amount"
        ).alias("total_sales"),

        F.avg(
            "unit_price"
        ).alias("average_unit_price")
    )
)

In [165]:
bridge_product_supplier = (
    product_suppliers
    .select(
        F.col("productid")
            .cast("long")
            .alias("product_id"),

        F.col("supplierid")
            .cast("long")
            .alias("supplier_id")
    )
    .dropDuplicates(
        ["product_id", "supplier_id"]
    )
)

In [167]:
write_parquet(dim_customer, "dim_customer")
write_parquet(dim_product, "dim_product")
write_parquet(dim_category, "dim_category")
write_parquet(dim_department, "dim_department")
write_parquet(dim_supplier, "dim_supplier")
write_parquet(dim_employee, "dim_employee")
write_parquet(dim_shipper, "dim_shipper")
write_parquet(dim_date, "dim_date")
write_parquet(dim_payment_method, "dim_payment_method")
write_parquet(dim_order_status, "dim_order_status")

Writing dim_customer -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_customer
Writing dim_product -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_product
Writing dim_category -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_category
Writing dim_department -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_department
Writing dim_supplier -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_supplier
Writing dim_employee -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_employee
Writing dim_shipper -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_shipper
Writing dim_date -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_date
Writing dim_payment_method -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_payment_method
Writing dim_order_status -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/dim_order_status


In [168]:
write_parquet(fact_order, "fact_order")
write_parquet(fact_order_detail, "fact_order_detail")
write_parquet(fact_payment, "fact_payment")
write_parquet(fact_shipment, "fact_shipment")
write_parquet(fact_customer_sales, "fact_customer_sales")
write_parquet(fact_product_sales, "fact_product_sales")

Writing fact_order -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/fact_order
Writing fact_order_detail -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/fact_order_detail
Writing fact_payment -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/fact_payment
Writing fact_shipment -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/fact_shipment
Writing fact_customer_sales -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/fact_customer_sales
Writing fact_product_sales -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/fact_product_sales


In [169]:
write_parquet(bridge_product_supplier,"bridge_product_supplier")

Writing bridge_product_supplier -> s3://aws-ecommerce-data-omar-2026/curated/ecommerce/bridge_product_supplier
